In [ ]:
# =============================================================
# ConstructionSight AI — YOLOv11 PPE Detection Trainer
# =============================================================


!pip install --no-cache-dir ultralytics==8.3.221 numpy==1.26.4 pandas matplotlib

import os, yaml, torch, pandas as pd, matplotlib.pyplot as plt
from ultralytics import YOLO
from pathlib import Path

# =============================================================
# Dataset setup
# =============================================================
DATA_ROOT = Path("/kaggle/input/contruction-ppe-dataset")  # dataset folder
YAML_IN   = DATA_ROOT / "data.yaml"
assert YAML_IN.exists(), f"❌ Dataset YAML not found: {YAML_IN}"

# Fix dataset paths if needed
with open(YAML_IN) as f:
    data = yaml.safe_load(f)
data["path"] = str(DATA_ROOT)
with open("ppe_data_fixed.yaml", "w") as f:
    yaml.safe_dump(data, f)
print("✅ Dataset YAML fixed.")

# =============================================================
# Model setup
# =============================================================
MODEL_PATH = "yolo11l.pt"     # YOLOv11-Large base model
PHASE1_EPOCHS = 2            # warm-up phase
PHASE2_EPOCHS = 5           # full training
BATCH = 8
IMGSZ = 1024                  # image size
PROJECT = "runs"
RUN_NAME = "FYP_PPE_PRO"

# =============================================================
# Phase 1 — frozen backbone
# =============================================================
print("\n🔹 [PHASE 1] Warming up backbone (frozen layers)...")
model = YOLO(MODEL_PATH)

results_phase1 = model.train(
    data="ppe_data_fixed.yaml",
    epochs=PHASE1_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    optimizer="AdamW",
    lr0=0.002,
    lrf=0.0005,
    cos_lr=True,
    pretrained=True,
    patience=20,
    freeze=10,
    mosaic=0.4,
    mixup=0.1,
    copy_paste=0.1,
    erasing=0.3,
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    translate=0.1, scale=0.4, degrees=5, shear=2.0,
    auto_augment="randaugment",
    amp=True,
    deterministic=True,
    project=PROJECT,
    name=f"{RUN_NAME}_PHASE1",
    plots=True,
    verbose=True
)

# =============================================================
# Phase 2 — full fine-tuning
# =============================================================
print("\n🔹 [PHASE 2] Full fine-tuning for maximum accuracy...")
model = YOLO(f"{PROJECT}/{RUN_NAME}_PHASE1/weights/best.pt")

results_phase2 = model.train(
    data="ppe_data_fixed.yaml",
    epochs=PHASE2_EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    optimizer="AdamW",
    lr0=0.001,
    lrf=0.0003,
    cos_lr=True,
    pretrained=True,
    patience=60,
    freeze=0,
    mosaic=0.7,
    mixup=0.15,
    copy_paste=0.2,
    erasing=0.4,
    hsv_h=0.02, hsv_s=0.7, hsv_v=0.5,
    translate=0.1, scale=0.5, degrees=10, shear=3.0,
    auto_augment="autoaugment",
    amp=True,
    deterministic=True,
    project=PROJECT,
    name=f"{RUN_NAME}_FINAL",
    plots=True,
    verbose=True
)

print("\n✅ Training completed for both phases!")

# =============================================================
# Validation
# =============================================================
print("\n🔹 Running final validation on test set...")
metrics = model.val(data="ppe_data_fixed.yaml", split="test")

print("\n📈 FINAL VALIDATION METRICS")
print(f" - mAP@50:     {metrics.box.map50:.4f}")
print(f" - mAP@50-95:  {metrics.box.map:.4f}")
print(f" - Precision:  {metrics.box.mp:.4f}")
print(f" - Recall:     {metrics.box.mr:.4f}")

# =============================================================
# Training curves
# =============================================================
run_dir = Path(f"{PROJECT}/{RUN_NAME}_FINAL")
results_csv = run_dir / "results.csv"
assert results_csv.exists(), f"⚠️ results.csv not found at {results_csv}"

df = pd.read_csv(results_csv)

# mAP curve
plt.figure(figsize=(6, 4))
plt.plot(df["epoch"], df["metrics/mAP50(B)"], color="teal", lw=2)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("mAP@50", fontsize=12)
plt.title("mAP@50 Over Epochs", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("map_curve.png", dpi=200)
plt.close()

# Loss curves
plt.figure(figsize=(6, 4))
plt.plot(df["epoch"], df["train/box_loss"], label="Box Loss", lw=2)
plt.plot(df["epoch"], df["train/cls_loss"], label="Cls Loss", lw=2)
if "train/dfl_loss" in df.columns:
    plt.plot(df["epoch"], df["train/dfl_loss"], label="DFL Loss", lw=2)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)
plt.title("Loss Curves", fontsize=14)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("loss_curve.png", dpi=200)
plt.close()

print("✅ Plots saved: map_curve.png, loss_curve.png")

# =============================================================
# Summary
# =============================================================
best_model_path = run_dir / "weights/best.pt"
print("\n🚀 TRAINING COMPLETE!")
print(f"📁 Best Model: {best_model_path}")
print(f"📊 mAP@50: {metrics.box.map50:.4f} | Precision: {metrics.box.mp:.4f} | Recall: {metrics.box.mr:.4f}")
print("✅ Model ready for ConstructionSight AI.")


In [ ]:
# Precision vs Recall
plt.figure(figsize=(6, 4))
plt.plot(df["metrics/precision(B)"], df["metrics/recall(B)"], color="purple", lw=2)
plt.xlabel("Precision", fontsize=12)
plt.ylabel("Recall", fontsize=12)
plt.title("Precision–Recall Curve", fontsize=14)
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("precision_recall_curve.png", dpi=200)
plt.close()

In [ ]:
# Learning rate progression
if "lr/pg0" in df.columns:
    plt.figure(figsize=(6, 4))
    plt.plot(df["epoch"], df["lr/pg0"], color="orange", lw=2)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Learning Rate", fontsize=12)
    plt.title("Learning Rate Schedule", fontsize=14)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig("lr_curve.png", dpi=200)
    plt.close()

In [ ]:
# Confidence vs mAP
if "metrics/confidence" in df.columns:
    plt.figure(figsize=(6, 4))
    plt.plot(df["metrics/confidence"], df["metrics/mAP50(B)"], color="darkgreen", lw=2)
    plt.xlabel("Confidence Threshold", fontsize=12)
    plt.ylabel("mAP@50", fontsize=12)
    plt.title("Confidence vs mAP@50", fontsize=14)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig("conf_map_curve.png", dpi=200)
    plt.close()

In [ ]:
# Epoch time trend
if "time/epoch" in df.columns:
    plt.figure(figsize=(6, 4))
    plt.plot(df["epoch"], df["time/epoch"], color="brown", lw=2)
    plt.xlabel("Epoch", fontsize=12)
    plt.ylabel("Seconds", fontsize=12)
    plt.title("Training Time per Epoch", fontsize=14)
    plt.grid(True, linestyle="--", alpha=0.6)
    plt.tight_layout()
    plt.savefig("epoch_time_curve.png", dpi=200)
    plt.close()

In [ ]:
# Combined overview
plt.figure(figsize=(8, 5))
plt.plot(df["epoch"], df["metrics/mAP50(B)"], label="mAP@50", lw=2)
plt.plot(df["epoch"], df["metrics/precision(B)"], label="Precision", lw=2)
plt.plot(df["epoch"], df["metrics/recall(B)"], label="Recall", lw=2)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Score", fontsize=12)
plt.title("Training Metrics Overview", fontsize=14)
plt.legend()
plt.grid(True, linestyle="--", alpha=0.6)
plt.tight_layout()
plt.savefig("metrics_overview.png", dpi=200)
plt.close()